# Overfit Single Source to Classical

This notebook demonstrates how to take a single rock audio source and overfit a model to transform it into classical style.

**Strategy:**
- Pick 1 source file from `rock_mel`
- Pair it with multiple classical targets for diversity
- Train with aggressive hyperparameters to force memorization
- Monitor loss and checkpoints

In [12]:
import torch
from pathlib import Path
import glob
from training.dataloader import RandomPairMelDataset
from torch.utils.data import DataLoader
from training.training import TrainingConfig, TrainingPipeline

print("="*70)
print("SETUP: Imports Complete")
print("="*70)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

SETUP: Imports Complete
✓ PyTorch version: 2.9.0+cpu
✓ CUDA available: False


In [13]:
# Setup paths
root = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")
data_dir = root / "data" / "output"

# Get file lists
rock_files = sorted(glob.glob(str(data_dir / "rock_mel" / "*.pt")))  # Source
classical_files = sorted(glob.glob(str(data_dir / "classical" / "*.pt")))  # Target

print("\n" + "="*70)
print("AVAILABLE DATA")
print("="*70)
print(f"Rock sources available: {len(rock_files)}")
print(f"Classical targets available: {len(classical_files)}")
print(f"\nFirst 3 rock files:")
for f in rock_files[:3]:
    print(f"  - {Path(f).name}")
print(f"\nFirst 3 classical files:")
for f in classical_files[:3]:
    print(f"  - {Path(f).name}")


AVAILABLE DATA
Rock sources available: 787
Classical targets available: 413

First 3 rock files:
  - 00000_instrum_mel.pt
  - 00001_instrum_mel.pt
  - 00002_instrum_mel.pt

First 3 classical files:
  - 00000_instrum_mel.pt
  - 00001_instrum_mel.pt
  - 00002_instrum_mel.pt


In [ ]:
# SELECT SINGLE SOURCE
source_idx = 0  # Change this to pick different sources (0-N)
single_source = rock_files[source_idx]

# Create training pairs: repeat single source, vary classical targets
source_files = [single_source] * len(classical_files)  # Repeat for all classical targets
target_files = classical_files

print("\n" + "="*70)
print("SELECTED SOURCE & TARGETS")
print("="*70)
print(f"Selected source file:")
print(f"  → {Path(single_source).name}")
print(f"\nTraining pairs: {len(source_files)}")
print(f"  Source: [same] × {len(source_files)}")
print(f"  Targets: {len(classical_files)} different classical files")
print(f"\nTarget classical files (first 5):")
for f in target_files[:5]:
    print(f"  - {Path(f).name}")


SELECTED SOURCE & TARGETS
Selected source file:
  → 00000_instrum_mel.pt

Training pairs: 413
  Source: [same] × 413
  Targets: 413 different rock files

Target rock files (first 5):
  - 00000_instrum_mel.pt
  - 00001_instrum_mel.pt
  - 00002_instrum_mel.pt
  - 00003_instrum_mel.pt
  - 00004_instrum_mel.pt


In [14]:
# Load sample to check shapes
sample_source = torch.load(single_source)
sample_target = torch.load(target_files[0])

print("\n" + "="*70)
print("DATA SAMPLE CHECK")
print("="*70)

# Handle dict vs tensor
if isinstance(sample_source, dict):
    for k in ('mel', 'spec', 'melspec', 'x'):
        if k in sample_source:
            sample_source = sample_source[k]
            break

if isinstance(sample_target, dict):
    for k in ('mel', 'spec', 'melspec', 'x'):
        if k in sample_target:
            sample_target = sample_target[k]
            break

print(f"Source shape: {sample_source.shape}")
print(f"Target shape: {sample_target.shape}")
print(f"Source dtype: {sample_source.dtype}")
print(f"Source min/max: [{sample_source.min():.3f}, {sample_source.max():.3f}]")
print(f"Target min/max: [{sample_target.min():.3f}, {sample_target.max():.3f}]")


DATA SAMPLE CHECK
Source shape: torch.Size([1, 100, 2811])
Target shape: torch.Size([1, 100, 2813])
Source dtype: torch.float32
Source min/max: [0.000, 4785.562]
Target min/max: [0.000, 4070.261]


In [15]:
# TRAINING CONFIG FOR OVERFITTING
config = TrainingConfig(
    num_epochs=100,           # More epochs to force learning
    batch_size=2,            # Small batch
    learning_rate=5e-4,      # Higher LR for faster convergence
    weight_decay=1e-5,       # Reduce regularization
    grad_clip_norm=1.0,      # Prevent explosion
    checkpoint_interval=1,   # Save every epoch
    checkpoint_dir="checkpoints/overfit_test",
    device="cuda" if torch.cuda.is_available() else "cpu",
    max_time=1024,           # Max mel time steps
    patch_height=10,
    patch_width=16,
    embed_dim=128,           # Smaller model
    num_blocks=2,            # Fewer blocks
    num_heads=2,             # Fewer heads
    hidden_dim=512,
    dropout=0.1,
    num_genres=2,            # Rock vs Classical
    in_channels=1,
)

print("\n" + "="*70)
print("TRAINING CONFIG")
print("="*70)
print(f"Epochs: {config.num_epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.learning_rate}")
print(f"Device: {config.device}")
print(f"Model size: embed_dim={config.embed_dim}, blocks={config.num_blocks}, heads={config.num_heads}")
print(f"Checkpoint dir: {config.checkpoint_dir}")


TRAINING CONFIG
Epochs: 100
Batch size: 2
Learning rate: 0.0005
Device: cpu
Model size: embed_dim=128, blocks=2, heads=2
Checkpoint dir: checkpoints/overfit_test


In [16]:
# Initialize pipeline
pipeline = TrainingPipeline(config)

print("\n" + "="*70)
print("PIPELINE INITIALIZED")
print("="*70)
print(f"✓ Model loaded to {config.device}")
print(f"✓ Optimizer: AdamW")
print(f"✓ Scheduler: ReduceLROnPlateau")

# Count parameters
total_params = sum(p.numel() for p in pipeline.flow.parameters())
trainable_params = sum(p.numel() for p in pipeline.flow.parameters() if p.requires_grad)
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")


PIPELINE INITIALIZED
✓ Model loaded to cpu
✓ Optimizer: AdamW
✓ Scheduler: ReduceLROnPlateau

Model parameters:
  Total: 1,488,928
  Trainable: 1,488,928


In [17]:
# Setup dataloader
loader = pipeline.setup_data(source_files, target_files)

print("\n" + "="*70)
print("DATALOADER READY")
print("="*70)
print(f"Total batches per epoch: {len(loader)}")
print(f"Batch size: {config.batch_size}")
print(f"Total samples: {len(source_files)}")

# Test batch
batch = next(iter(loader))
x0_batch, x1_batch, mask_batch = batch
print(f"\nBatch shapes:")
print(f"  x0 (source/rock): {x0_batch.shape} → {tuple(x0_batch.shape)}")
print(f"  x1 (target/classical): {x1_batch.shape}")
print(f"  mask: {mask_batch.shape}")
print(f"\n✓ Data ready for training!")


DATALOADER READY
Total batches per epoch: 207
Batch size: 2
Total samples: 413

Batch shapes:
  x0 (source/rock): torch.Size([2, 1, 100, 1024]) → (2, 1, 100, 1024)
  x1 (target/classical): torch.Size([2, 1, 100, 1024])
  mask: torch.Size([2, 1, 1, 1024])

✓ Data ready for training!


## Training Loop

Now we'll train the model. This will overfit the single rock source into classical style.

In [ ]:
# START TRAINING
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Training {Path(single_source).name} → Classical style")
print(f"Total epochs: {config.num_epochs}")
print(f"Expected batches: {len(loader) * config.num_epochs}")
print("="*70 + "\n")

try:
    pipeline.train(loader)
    print("\n" + "="*70)
    print("✓ TRAINING COMPLETED SUCCESSFULLY")
    print("="*70)
except KeyboardInterrupt:
    print("\n" + "="*70)
    print("⚠ Training interrupted by user")
    print("="*70)
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()


STARTING TRAINING
Training 00000_instrum_mel.pt → Classical style
Total epochs: 100
Expected batches: 20700


Epoch 1/100 started
Starting batch shapes:
  x0: torch.Size([2, 1, 100, 1024])
  x1: torch.Size([2, 1, 100, 1024])
  mask: torch.Size([2, 1, 1, 1024])
Epoch 1/100 - Step 10/207 - Loss: 14508.5195
Epoch 1/100 - Step 20/207 - Loss: 17866.6895
Epoch 1/100 - Step 30/207 - Loss: 28757.9609
Epoch 1/100 - Step 40/207 - Loss: 16663.2246
Epoch 1/100 - Step 50/207 - Loss: 14148.2197
Epoch 1/100 - Step 60/207 - Loss: 13923.6074
Epoch 1/100 - Step 70/207 - Loss: 14195.1836
Epoch 1/100 - Step 80/207 - Loss: 13711.5752
Epoch 1/100 - Step 90/207 - Loss: 13350.2490
Epoch 1/100 - Step 100/207 - Loss: 21662.7422
Epoch 1/100 - Step 110/207 - Loss: 13880.7783
Epoch 1/100 - Step 120/207 - Loss: 15412.6885
Epoch 1/100 - Step 130/207 - Loss: 13947.4365
Epoch 1/100 - Step 140/207 - Loss: 14606.6924
Epoch 1/100 - Step 150/207 - Loss: 18069.8203
Epoch 1/100 - Step 160/207 - Loss: 13634.2666
Epoch 1/100

: 

In [11]:
# Check checkpoints
import os

checkpoint_dir = Path(config.checkpoint_dir)
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("*.pt"))
    print("\n" + "="*70)
    print("SAVED CHECKPOINTS")
    print("="*70)
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / (1024**2)
        print(f"  {ckpt.name} ({size_mb:.1f} MB)")
    print(f"\nTotal checkpoints: {len(checkpoints)}")
    print(f"Latest: {checkpoints[-1].name if checkpoints else 'None'}")
else:
    print(f"\n⚠ Checkpoint directory not found: {checkpoint_dir}")


SAVED CHECKPOINTS
  checkpoint_epoch_005.pt (17.1 MB)
  checkpoint_epoch_010.pt (17.1 MB)

Total checkpoints: 2
Latest: checkpoint_epoch_010.pt


## Next Steps

After training:
1. Use the saved checkpoint to perform inference on your single rock source
2. Listen to the output to verify classical transformation
3. Adjust hyperparameters (learning rate, epochs) based on results
4. Try different source files to see varying results